In [ ]:
!pip install opendatasets -q

In [ ]:
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import opendatasets as od
from tensorflow.keras.regularizers import l2
from sklearn.metrics import classification_report

In [ ]:
od.download("https://www.kaggle.com/datasets/dineshpiyasamara/cats-and-dogs-for-classification")

In [ ]:
BATCH_SIZE = 32
IMAGE_SIZE = (128, 128)

In [ ]:
train_data_dir = "/content/cats-and-dogs-for-classification/cats_dogs/train"
test_data_dir = "/content/cats-and-dogs-for-classification/cats_dogs/test"

In [ ]:
train_data = tf.keras.utils.image_dataset_from_directory(
    train_data_dir,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    validation_split=0.1,
    subset="training",
    seed=42
)

validation_data = tf.keras.utils.image_dataset_from_directory(
    train_data_dir,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    validation_split=0.1,
    subset="validation",
    seed=42
)

test_data = tf.keras.utils.image_dataset_from_directory(
    test_data_dir,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE
)

In [ ]:
class_names = train_data.class_names
print("Classes:", class_names)

In [ ]:
for image_batch, label_batch in train_data.take(1):
  print(image_batch.shape)
  print(label_batch.shape)

In [ ]:
plt.figure(figsize=(10, 4))
for image, label in train_data.take(1):
  for i in range(10):
    ax = plt.subplot(2, 5, i+1)
    plt.imshow(image[i].numpy().astype('uint8'))
    plt.title(class_names[label[i]])
    plt.axis('off')

In [ ]:
for image, label in train_data.take(1):
  for i in range(1):
    print(image)

In [ ]:
# Feature Scaling
train_data      = train_data.map(lambda x, y: (x/255.0, y))
validation_data = validation_data.map(lambda x, y: (x/255.0, y))
test_data       = test_data.map(lambda x, y: (x/255.0, y))

In [ ]:
for image, label in train_data.take(1):
  for i in range(1):
    print(image)

In [ ]:
# Data Augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

In [ ]:
# Load Pre-Trained Model
base_model = tf.keras.applications.Xception(
    include_top=False,
    input_shape=(128, 128, 3),
    weights='imagenet'
)
base_model.trainable = False
base_model.summary()

In [ ]:
# Model Building
model = tf.keras.Sequential([
    data_augmentation,
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.summary()

In [ ]:
# Model Compiling
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Callbacks
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath="/content/drive/MyDrive/Machine Learning/Image Based/Cats_Dogs_IMG_Classification/transfer_learning_model.keras",
    save_best_only=True
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
    checkpoint_cb
]

In [ ]:
# Model Training (Phase 1 – Transfer Learning)
history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=15,
    callbacks=callbacks
)

In [ ]:
# Fine-Tuning (Phase 2 – Unfreeze last layers)
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
# Model Evaluation
y_true = []
y_pred = []

for X, y in test_data:
    preds = model.predict(X, verbose=0)
    preds = (preds > 0.5).astype(int).flatten()

    y_true.extend(y.numpy())
    y_pred.extend(preds)

print(classification_report(y_true, y_pred, target_names=class_names))